# 🧪 YOLO11 통합 노트북 (Colab)

Albumentations 오프라인 증강을 합친 최종본.

**사용법:** 

① `2. 설정` + `4. 증강 설정` 두 곳만 수정 

② 위에서 아래로 `모두 실행`.

모든 학습/증강 파라미터는 **한 곳(config·AUG_CONFIG)** 에서만 정하도록 정리했습니다.

## 0. 설치


In [ ]:
# 저장소 루트에서 실행하거나 PILL_PROJECT_ROOT 환경변수로 지정합니다.
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("PILL_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "requirements.txt").is_file(), (
    "저장소 루트에서 노트북을 실행하거나 PILL_PROJECT_ROOT를 지정하세요: "
    f"{PROJECT_ROOT}"
)

%pip install -q -r {PROJECT_ROOT / "requirements.txt"}


## 1. Google Drive 연결 · 공통 import


In [ ]:
from pathlib import Path
import sys, os, json, shutil, gc
import torch
from omegaconf import OmegaConf
from ultralytics import YOLO

# --- Colab이면 드라이브 연결 (로컬이면 이 블록만 건너뛰세요) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

## 2. ⚙️ 설정 — 경로·모델·온라인증강 (여기만)


In [ ]:
# ============================================================
# ★★★ 설정 (경로 + 실험) — 각자 환경에 맞게 수정 ★★★
# ============================================================

# --- (A) 경로: 본인 드라이브 구조에 맞게 ---
YOLO_DATASET_DIR = PROJECT_ROOT / "data" / "processed"    # ★ 증강 전 "원본" YOLO 데이터셋(images/labels/data.yaml)
RAW_DATASET_ROOT = PROJECT_ROOT / "data" / "dataset" / "cleaning_data"

USE_LOCAL_COPY = False   # Colab: 데이터셋을 /content로 두어 학습 빠르게 (강력 권장)

# --- (B) config (모든 학습 설정의 단일 소스) ---
CONFIG = {
    "project": {"seed": 42},
    "paths": {
        "project_root":    str(PROJECT_ROOT),
        "dataset_root":    str(RAW_DATASET_ROOT),
        "annotation_dir":  str(RAW_DATASET_ROOT / "train_annotations"),
        "test_image_dir":  str(RAW_DATASET_ROOT / "test_images"),
        "yolo_dataset_dir": str(YOLO_DATASET_DIR),
        "weights_path":    "",
        "submission_path": str(PROJECT_ROOT / "outputs" / "submission" / "submission-checker.csv"),
    },

    # ▼▼▼ 실험할 때 바꾸는 부분 (여기 한 곳) ▼▼▼
    "model": {"name": "yolo11s.pt"},          # yolo11s / yolo11m ...
    "train": {
        "epochs": 50, "imgsz": 1024, "batch": 16,
        "workers": 2,          # Colab에서 학습이 첫 배치에서 멈추면 0
        "patience": 10,        # val 개선 없을 때 대기 에포크
        "checkpoint_name": "yolo11s-exp",
    },
    "yolo_augmentation": {     # YOLO 내장(온라인) 증강 — 오프라인 증강과 겹치면 낮게
        "hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.1,
        "degrees": 0.0, "translate": 0.0, "scale": 0.0, "shear": 0.5, "perspective": 0.0,
        "flipud": 0.0, "fliplr": 0.0,
        "mosaic": 0.0, "mixup": 0.0, "copy_paste": 0.0, "erasing": 0.0,
    },
    # ▲▲▲

    "dataset": {"image_dir_name": "train_images", "annotation_dir_name": "train_annotations",
                "label_offset": 0, "strict": False, "validate_image_size": True,
                "train_ratio": 0.8, "val_ratio": 0.1, "test_ratio": 0.1},
    "evaluation": {"imgsz": 1024, "plots": True},
    "inference":  {"imgsz": 1024, "conf": 0.25, "max_det": 4},   # conf 낮추면 recall↑
    "wandb": {"enabled": True, "mode": "online", "project": "pill-object-detection",
              "entity": "diokim17", "run_name": None, "tags": ["yolo11s"]},
}
cfg = OmegaConf.create(CONFIG)

# src.yolo 패키지 import (PROJECT_ROOT + src/yolo 둘 다 등록)
SRC_ROOT = PROJECT_ROOT / "src" / "yolo"
for p in (str(PROJECT_ROOT), str(SRC_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
from src.yolo.evaluation import evaluate_yolo, print_evaluation
from src.yolo.submission import (list_test_images, run_inference_sanity_check,
                                 create_submission, validate_submission)
from src.yolo.yolo_mapping import build_yolo_to_category_id
from src.yolo.runtime import set_seed, resolve_device, print_runtime_info

print("설정 로드 | 모델:", cfg.model.name, "| epochs:", cfg.train.epochs, "| imgsz:", cfg.train.imgsz)


## 3. 런타임 (시드 · 디바이스)


In [ ]:
set_seed(cfg.project.seed)
device = resolve_device()          # GPU 있으면 0, 없으면 "cpu"
print_runtime_info(cfg.project.seed, device)

### 3-1. 호환 변수 (증강 셀이 쓰는 이름 맞춰주기)


In [ ]:
import wandb
SEED   = cfg.project.seed
DEVICE = device
DATASET_DIR  = Path(cfg.paths.yolo_dataset_dir)      # 증강 전 "원본" YOLO 데이터셋 (증강의 src)
RUNTIME_YAML = DATASET_DIR / "data.yaml"             # 증강 안 쓸 때 fallback
assert (DATASET_DIR / "images" / "train").is_dir(), f"원본 데이터셋 없음: {DATASET_DIR}"
print("증강 src(원본):", DATASET_DIR)

## 4. 오프라인 증강 (Albumentations) — ★ 수정하는 곳 ★


In [ ]:
# ============================================================
# 4-1. [D] Albumentations 오프라인 증강 설정
#      ★★★ 팀원이 수정하는 곳은 여기입니다 ★★★
# ============================================================
import sys, importlib
sys.path.insert(0, str(PROJECT_ROOT / "src"))       # pill_transforms.py 위치
sys.modules.pop("pill_transforms", None)             # 옛 버전 캐시 제거
import pill_transforms as pt
importlib.reload(pt)
print("pill_transforms:", pt.__file__)

# ────────────────────────────────────────────────────────────
# [D-1] 증강을 쓸지 / 몇 배로 늘릴지
# ────────────────────────────────────────────────────────────
ALBU_ENABLE   = True    # False면 증강 없이 원본 데이터셋으로 바로 학습합니다
AUG_PER_IMAGE = 1       # ★ 증강 배수 조절 ★ 원본 1장당 만들 증강본 개수
                        #   권장 1~3 / 한계 0~10
                        #   최종 학습 장수 = 원본 x (1 + AUG_PER_IMAGE)
                        #   예) 원본 1,000장 + AUG_PER_IMAGE=2 → 3,000장
                        #   ⚠️ 디스크 사용량과 1에포크 시간이 그대로 (1+N)배 됩니다
AUG_KEEP_ORIGINAL = True  # 원본도 학습에 포함(권장 True).
                          # False면 모델이 "증강된 그림"만 보게 되어
                          # 실제 테스트 이미지와 분포가 어긋납니다
AUG_MAX_IMAGES = None     # None=전체. 숫자를 넣으면 앞의 N장만 처리(빠른 테스트용)

# ────────────────────────────────────────────────────────────
# [D-2] 어떤 효과를 켤지 — True/False 로만 켜고 끕니다
#       ⚠️ 예전처럼 p=0.4 같은 확률로 "걸릴지 말지" 랜덤하지 않습니다.
#          ON 이면 만드는 이미지마다 반드시 들어갑니다.
#          (세기는 지정 범위 안에서 매번 다르게 뽑힙니다. 안 그러면
#           N장을 만들어도 전부 똑같은 이미지가 나옵니다.)
# ────────────────────────────────────────────────────────────
AUG_CONFIG = dict(
    # ── 기하 변환 ─────────────────────────────────────────
    CROP_ENABLE       = False,  # bbox 보존 랜덤 크롭.
                                #  ⚠️ 정사각으로 잘라 종횡비가 찌그러지고
                                #     알약 "크기" 단서도 깨집니다 → OFF 권장
    CROP_EROSION_RATE = 0.0,    #  권장 0.0~0.2 / 한계 0.0~0.5

    RESIZE_ENABLE     = False,  # 리사이즈+패딩.
                                #  ⚠️ YOLO가 학습 때 또 letterbox 하므로
                                #     여기서 하면 두 번 리사이즈 = 각인 뭉개짐
                                #     → 오프라인 증강에서는 OFF 권장
    IMAGE_SIZE        = 960,    #  RESIZE_ENABLE=True 일 때만 의미. 권장 640~960
    PAD_VALUE         = 0,      #  여백 색(0=검정). 이 데이터셋 배경은 연회색이라
                                #  200~220 을 주면 더 자연스럽습니다

    HFLIP_ENABLE      = False,  # 좌우반전 ★ 각인 글자가 거울상 → OFF 고정 권장
    VFLIP_ENABLE      = False,  # 상하반전 ★ 같은 이유로 OFF 고정 권장

    ROTATE_ENABLE     = True,   # 회전 (촬영각 70/75/90도 변화 모사)
    ROTATE_LIMIT      = 5,     #  ±N도. 권장 10~20 / 한계 0~45
                                #  ⚠️ 45도 넘기면 축정렬 bbox가 알약보다 훨씬 커집니다

    # ── 색·명암 (예전에는 OneOf로 셋 중 하나만 걸렸음 → 이제 각각 독립) ──
    BRIGHTNESS_ENABLE = False,   # 밝기/대비
    BRIGHTNESS_LIMIT  = 0.2,    #  권장 0.15~0.30 / 한계 0.0~0.5
    CONTRAST_LIMIT    = 0.2,    #  권장 0.15~0.30 / 한계 0.0~0.5
                                #  ⚠️ 0.5 넘기면 흰 알약이 배경에 묻힙니다

    HSV_ENABLE        = True,   # 색상/채도/명도
    HUE_SHIFT_LIMIT   = 0,     #  ⚠️⚠️ 알약 색이 곧 클래스입니다. 권장 5~10 / 한계 15
    SAT_SHIFT_LIMIT   = 0,     #  권장 10~25 / 한계 40
    VAL_SHIFT_LIMIT   = 0,     #  권장 10~20 / 한계 40

    CLAHE_ENABLE      = True,   # 그림자 보정(국소 대비 평탄화)
    CLAHE_CLIP_LIMIT  = 2.0,    #  권장 2.0~3.0 / 한계 1.0~4.0
                                #  ⚠️ 넘기면 노이즈까지 증폭

    # ── 화질 열화 (예전에는 OneOf로 둘 중 하나 → 이제 각각 독립) ──
    NOISE_ENABLE      = True,   # 가우시안 노이즈(센서 노이즈)
    NOISE_STD_MIN     = 0.02,   #  권장 0.02~0.10 / 한계 0.20
    NOISE_STD_MAX     = 0.03,   #  ⚠️ 0.2 넘기면 각인이 노이즈에 묻힙니다

    BLUR_ENABLE       = False,   # 모션 블러(흔들림)
    BLUR_LIMIT        = 3,      #  홀수만. 권장 3 / 한계 3~9
                                #  ⚠️ 7 이상이면 각인이 사실상 사라집니다

    # ── bbox 필터 ─────────────────────────────────────────
    MIN_VISIBILITY    = 0.2,    # 잘린 뒤 이만큼 안 남으면 라벨 삭제. 권장 0.2~0.4
    MIN_AREA          = 4.0,    # 최소 bbox 픽셀 면적. 권장 4~64

    # ── GAN 스타일 증강 (학습된 생성기가 있을 때만) ────────
    GAN_ENABLE        = False,
    GAN_MODEL_PATH    = str(PROJECT_ROOT / "models" / "generator.onnx"),
    GAN_STRENGTH      = 0.5,    # 원본과 섞는 비율. 권장 0.3~0.6
)

# GAN을 켤 때만 필요한 전역 설정 (모델 형식 관련)
if AUG_CONFIG["GAN_ENABLE"]:
    pt.GAN_MODEL_PATH    = AUG_CONFIG["GAN_MODEL_PATH"]
    pt.GAN_INPUT_SIZE    = 512        # 생성기 입력 크기. 권장 256~512
    pt.GAN_INPUT_RANGE   = "tanh"     # "tanh"(-1~1) | "sigmoid"(0~1)
    pt.GAN_CHANNEL_ORDER = "rgb"      # "rgb" | "bgr"
    pt.GAN_DEVICE        = "cuda" if DEVICE != "cpu" else "cpu"
    pt.GANGenerator.clear_cache()
    print(pt.describe_gan()); print()
    assert pt.check_gan_model(), "GAN 모델을 쓸 수 없습니다. 위 메시지를 확인하세요."

# ────────────────────────────────────────────────────────────
# [D-3] 산출물 경로 (README_AUGMENTATION.md 의 폴더 구조와 동일)
# ────────────────────────────────────────────────────────────
AUG_TAG        = f"albu_x{1 + AUG_PER_IMAGE}"            # 실험 구분용 태그
AUG_DATASET_DIR = PROJECT_ROOT / "data" / f"processed_{AUG_TAG}"   # 증강 데이터셋
AUG_REPORT_DIR  = PROJECT_ROOT / "outputs" / "augmentation" / AUG_TAG  # 리포트·검수이미지

print(pt.describe_augmentation(AUG_CONFIG))
print()
print(f"증강 배수      원본 x {1 + AUG_PER_IMAGE if AUG_KEEP_ORIGINAL else AUG_PER_IMAGE}")
print(f"증강 데이터셋  {AUG_DATASET_DIR}")
print(f"리포트 폴더    {AUG_REPORT_DIR}")


### 4-b. 증강 미리보기 (파일 생성 전 3장 확인)


In [ ]:
# ============================================================
# 4-1-b. 증강 미리보기 (파일 생성 없이 3장만 확인)
# ============================================================
from IPython.display import Image as IPyImage, display

_preview_paths = pt.preview_augmentation(
    src_root=DATASET_DIR,
    out_dir=AUG_REPORT_DIR / "preview",
    split="train",
    n_images=3,                    # 몇 장을 볼지
    n_aug=AUG_PER_IMAGE,           # 한 장당 몇 개의 증강본을 볼지
    config=AUG_CONFIG,
    seed=SEED,
)
for _p in _preview_paths:
    display(IPyImage(filename=_p, width=1400))


### 4-c. 증강 데이터셋 생성 (+ 수량 리포트)


In [ ]:
# ============================================================
# 4-1-c. 증강 데이터셋 생성 (+ 전체/증강 데이터 개수 집계)
# ============================================================

# ---------- (1) 원본 데이터 현황 ----------
orig_stat = pt.print_dataset_stats(DATASET_DIR, title="원본 데이터셋")
N_TRAIN_ORIG = orig_stat.get("train", {}).get("images", 0)
print(f"\n★ 원본 train 이미지: {N_TRAIN_ORIG:,} 장\n")

# ---------- (2) 증강 실행 ----------
TRAIN_YAML = RUNTIME_YAML          # 기본값: 증강 안 쓸 때는 원본 yaml
AUG_STATS = None

if ALBU_ENABLE and AUG_PER_IMAGE > 0:
    AUG_STATS = pt.augment_dataset(
        src_root=DATASET_DIR,
        dst_root=AUG_DATASET_DIR,
        n_aug=AUG_PER_IMAGE,            # ★ 증강 배수
        split="train",
        config=AUG_CONFIG,              # ★ 어떤 효과를 켤지
        keep_original=AUG_KEEP_ORIGINAL,
        report_dir=AUG_REPORT_DIR,
        n_preview=8,                    # 검수용 비교 이미지 장수
        max_images=AUG_MAX_IMAGES,
        seed=SEED,
        overwrite=True,                 # 기존 출력 폴더를 지우고 새로 만듭니다
    )
    TRAIN_YAML = AUG_STATS["data_yaml"]

    # ---------- (3) 최종 요약 ----------
    print("\n" + "=" * 62)
    print("  최종 학습 데이터 요약")
    print("=" * 62)
    print(f"  원본 train        {AUG_STATS['n_source_images']:,} 장")
    print(f"  증강본            {AUG_STATS['n_augmented']:,} 장  "
          f"(1장당 {AUG_STATS['n_aug_per_image']}개)")
    print(f"  ★ 총 학습 데이터  {AUG_STATS['n_total']:,} 장  "
          f"= 원본의 {AUG_STATS['multiplier']}배")
    print(f"  ★ 총 bbox         {AUG_STATS['boxes_total']:,} 개")
    print("=" * 62)
else:
    print("Albumentations 증강 사용 안 함 — 원본 데이터셋으로 학습합니다.")

print("\n학습에 쓸 data.yaml:", TRAIN_YAML)


# --- 학습/평가에 쓸 data.yaml + config 연결 ---
data_yaml = Path(TRAIN_YAML)
cfg.paths.yolo_dataset_dir = str(data_yaml.parent)
print("→ 학습 data.yaml:", data_yaml)

### 4-d. 저장된 검수 이미지 (원본 | 증강본, bbox)


In [ ]:
# ============================================================
# 4-1-d. 저장된 검수 이미지 보기
# ============================================================
from IPython.display import Image as IPyImage, display
from pathlib import Path

_samples = sorted((AUG_REPORT_DIR / "samples").glob("*.png"))
print(f"{len(_samples)}장  ({AUG_REPORT_DIR / 'samples'})")
for _p in _samples[:5]:            # 전부 보려면 [:5] 를 지우세요
    display(IPyImage(filename=str(_p), width=1400))

# 리포트 내용도 노트북에서 바로 확인
_report = AUG_REPORT_DIR / "augmentation_report.md"
if _report.exists():
    from IPython.display import Markdown
    display(Markdown(_report.read_text(encoding="utf-8")))


## 5. (Colab) 학습 데이터셋 로컬 확인


In [ ]:
# 증강 결과는 이미 /content(로컬). 원본만 쓰는 경우(증강 OFF)엔 로컬로 복사.
if USE_LOCAL_COPY and IN_COLAB and not str(cfg.paths.yolo_dataset_dir).startswith("/content"):
    local = PROJECT_ROOT / "outputs" / "cache" / "yolo_dataset"
    if not local.exists():
        print("드라이브 → 로컬 복사 중... (1회)")
        shutil.copytree(cfg.paths.yolo_dataset_dir, str(local))
    cfg.paths.yolo_dataset_dir = str(local)
    data_yaml = local / "data.yaml"
print("학습 데이터셋:", cfg.paths.yolo_dataset_dir)


## 6. data.yaml 확인 · 경로 맞춤


In [ ]:
import yaml
data_yaml = Path(data_yaml)
PROCESSED_ROOT = data_yaml.parent
assert (PROCESSED_ROOT / "images" / "train").is_dir(), f"images/train 없음: {PROCESSED_ROOT}"

ycfg = yaml.safe_load(open(data_yaml, encoding="utf-8"))
ycfg["path"] = str(PROCESSED_ROOT.resolve())
if "nc" not in ycfg and "names" in ycfg:
    ycfg["nc"] = len(ycfg["names"])
yaml.safe_dump(ycfg, open(data_yaml, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
n_names = len(ycfg["names"])
print("클래스 수:", n_names, "| data.yaml:", data_yaml)

## 7. 클래스 역매핑 (YOLO id → 원본 category_id)


In [ ]:
# 원본 annotation에서 YOLO class ID → 원본 category_id 매핑을 재생성합니다.
yolo_to_category_id = build_yolo_to_category_id(Path(cfg.paths.annotation_dir))
assert yolo_to_category_id, f"category mapping 생성 실패: {cfg.paths.annotation_dir}"
print("매핑 클래스 수:", len(yolo_to_category_id))
assert len(yolo_to_category_id) == n_names, f"매핑 {len(yolo_to_category_id)} != data.yaml {n_names}"
print("예시:", list(yolo_to_category_id.items())[:3])


## 8. 학습 (설정은 섹션 2·4에서 옴)


In [ ]:
RUN_NAME = f"{cfg.train.checkpoint_name}_{AUG_TAG}" if (ALBU_ENABLE and AUG_PER_IMAGE > 0) else cfg.train.checkpoint_name
_aug_on = [k.replace("_ENABLE", "") for k, v in AUG_CONFIG.items() if k.endswith("_ENABLE") and v]

wandb.init(project=cfg.wandb.project, name=RUN_NAME, entity=cfg.wandb.entity, job_type="train",
           tags=list(cfg.wandb.tags) + (["albu"] if ALBU_ENABLE else []),
           notes=f"{cfg.train.epochs}ep patience{cfg.train.patience} | albu x{AUG_PER_IMAGE} [{','.join(_aug_on)}]")

model = YOLO(cfg.model.name)
results = model.train(
    data=str(data_yaml),
    epochs=cfg.train.epochs, patience=cfg.train.patience,
    imgsz=cfg.train.imgsz, batch=cfg.train.batch,
    seed=cfg.project.seed, deterministic=True, device=device, workers=cfg.train.workers,
    project=str(PROJECT_ROOT / "outputs/yolo"), name=RUN_NAME, exist_ok=True,
    **dict(cfg.yolo_augmentation),        # 온라인 증강도 config에서
)
best_weights = Path(results.save_dir) / "weights" / "best.pt"
cfg.paths.weights_path = str(best_weights)
print("결과 폴더:", results.save_dir, "| best:", best_weights)

### 8-b. 증강 후 데이터 상태 시각화 (train_batch)


In [ ]:
# ============================================================
# 증강 후 데이터 상태 시각화 (Ultralytics가 학습 중 저장한 실제 증강 배치)
# ============================================================
from IPython.display import Image as IPyImage, display
from pathlib import Path

run_dir = Path(results.save_dir)   # 학습 결과 폴더
shown = False
for name in ["train_batch0.jpg", "train_batch1.jpg", "train_batch2.jpg"]:
    p = run_dir / name
    if p.exists():
        print(f"── {name} (증강 적용된 학습 배치) ──")
        display(IPyImage(filename=str(p)))
        shown = True
if not shown:
    print("train_batch*.jpg 가 아직 없어요. 학습을 최소 1 epoch 이상 돌렸는지 확인하세요.")

## 9. best.pt 백업 (로컬 → Drive)


In [ ]:
# 로컬(/content)에 저장된 결과는 런타임 종료 시 사라지므로 Drive로 백업
dst = PROJECT_ROOT / "outputs" / "yolo" / f"{cfg.train.checkpoint_name}_best.pt"
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(str(best_weights), str(dst))
print("Drive 백업:", dst)

## 10. 검증 (mAP) — val · test


In [ ]:
for split in ["val", "test"]:
    res = evaluate_yolo(best_weights, data_yaml=data_yaml, split=split,
                        imgsz=cfg.evaluation.imgsz, device=device, plots=cfg.evaluation.plots)
    print_evaluation(res, prefix=split)

## 11. Kaggle 제출 파일 생성 · 검증


In [ ]:
test_dir = Path(cfg.paths.test_image_dir)

# 제출 전 1장 sanity 체크
run_inference_sanity_check(best_weights, test_image_dir=test_dir,
                           yolo_to_category_id=yolo_to_category_id,
                           imgsz=cfg.inference.imgsz, conf=cfg.inference.conf,
                           max_det=cfg.inference.max_det, device=device)

submission_path = Path(cfg.paths.submission_path)
submission_path = create_submission(best_weights, test_image_dir=test_dir,
                                    submission_path=submission_path,
                                    yolo_to_category_id=yolo_to_category_id,
                                    imgsz=cfg.inference.imgsz, conf=cfg.inference.conf,
                                    max_det=cfg.inference.max_det, device=device,
                                    clear_cache_every=100)

df = validate_submission(submission_path,
                         valid_category_ids=yolo_to_category_id.values(),
                         expected_image_count=len(list_test_images(test_dir)))
print("제출:", submission_path, "| 행:", len(df))
display(df.head())

## 12. (선택) 예측 시각화


In [ ]:
import math, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from matplotlib.patches import Rectangle
from PIL import Image
import pandas as pd

!apt-get -qq install -y fonts-nanum >/dev/null 2>&1
for fp in fm.findSystemFonts():
    if "Nanum" in fp: fm.fontManager.addfont(fp)
plt.rcParams["font.family"] = "NanumGothic"; plt.rcParams["axes.unicode_minus"] = False

cat2name = {v: k for k, v in zip(ycfg["names"].values(), yolo_to_category_id.values())} \
           if isinstance(ycfg["names"], dict) else {}
sub = pd.read_csv(submission_path)
ids = sorted(sub["image_id"].unique())[:20]          # 앞 20장만 미리보기
out_dir = PROJECT_ROOT / "outputs" / "predictions" / "pred_vis"; out_dir.mkdir(exist_ok=True)

for page in range(math.ceil(len(ids)/10)):
    chunk = ids[page*10:(page+1)*10]
    fig, axes = plt.subplots(2, 5, figsize=(25, 11)); axes = axes.ravel()
    for k, iid in enumerate(chunk):
        ax = axes[k]
        try: ax.imshow(Image.open(test_dir / f"{iid}.png").convert("RGB"))
        except Exception: ax.axis("off"); continue
        for j, (_, r) in enumerate(sub[sub.image_id == iid].iterrows()):
            ax.add_patch(Rectangle((r.bbox_x, r.bbox_y), r.bbox_w, r.bbox_h,
                                   fill=False, edgecolor=plt.cm.tab10(j % 10), lw=2))
            ax.text(r.bbox_x, r.bbox_y - 5, f"{int(r.category_id)} {r.score:.2f}", fontsize=8, color="red")
        ax.set_title(f"[{iid}] 검출 {len(sub[sub.image_id==iid])}"); ax.axis("off")
    for k in range(len(chunk), 10): axes[k].axis("off")
    fig.savefig(out_dir / f"pred_{page+1:02d}.png", dpi=120, bbox_inches="tight"); plt.close(fig)
print("시각화 저장:", out_dir)
